# Evaluasi Prediksi Waktu Tunggu Kantin
### Random Forest Regression | Notebook Kaggle

Notebook ini membantu membandingkan **Random Forest Regression**, perkiraan sederhana berdasarkan **median**, dan **Linear Regression**. Ukuran utama adalah MAE, yaitu rata-rata selisih absolut prediksi dengan waktu sebenarnya dalam menit.

**Penting:** notebook awalnya menggunakan data **simulasi**, sehingga hasil yang keluar hanya menunjukkan cara kerja kode, bukan bukti akurasi di kantin. Evaluasi yang relevan memerlukan data nyata dari kantin. Model dalam notebook ini adalah eksperimen terpisah dan tidak otomatis mengganti model pada aplikasi.

## Cara menggunakan di Kaggle
1. Impor notebook `.ipynb` ini ke Kaggle, lalu buka editor notebook.
2. Untuk mencoba dulu, biarkan `SUMBER_DATA = "simulasi"` dan jalankan semua sel dari atas ke bawah.
3. Untuk memakai `data_kantin_simulasi.csv` dari menu kantin atau CSV hasil pencatatan nyata, lampirkan file CSV sebagai input notebook. Lihat lokasi file pada panel input Kaggle, lalu ubah `SUMBER_DATA` menjadi `"csv"` dan isi `PATH_CSV` sesuai lokasi file, biasanya di bawah `/kaggle/input/`.
4. Jalankan ulang seluruh sel secara berurutan. Perhatikan keterangan sumber data di hasil dan bandingkan MAE pada data uji.

Data contoh dari menu kantin berlabel `sumber_data=simulasi`. Jangan gabungkan dengan catatan nyata. Pilihan menu pada CSV simulasi disampling merata untuk menguji semua menu. Jam ramai, jumlah petugas, dan rentang penyiapan merupakan asumsi pembuat data.

Dataset nyata belum disertakan. Tidak diperlukan kredensial di notebook. Pastikan semua waktu dicatat dalam format dan zona waktu yang sama.

## Format CSV untuk data nyata

Satu baris mewakili satu pesanan yang **hanya berisi satu jenis menu**. Jumlah porsi tetap boleh lebih dari satu. Contoh format (angka contoh, bukan data pengamatan):

| waktu_pesan | waktu_siap | menu | jumlah | antrean_di_depan |
|---|---|---|---:|---:|
| 2026-11-16 12:00:00 | 2026-11-16 12:18:00 | Nasi Goreng | 2 | 4 |
| 2026-11-16 12:03:00 | 2026-11-16 12:10:00 | Mie Ayam | 1 | 2 |

- `waktu_pesan`: saat pesanan diterima; `waktu_siap`: saat makanan siap diambil, **bukan** saat diambil pelanggan.
- `menu`: nama menu; `jumlah`: jumlah porsi berupa bilangan bulat positif.
- `antrean_di_depan`: jumlah pesanan yang masih menunggu atau diproses saat pesanan diterima, berupa bilangan bulat nonnegatif.
- Target dihitung otomatis: **waktu_siap - waktu_pesan** dalam menit. Waktu siap dipakai untuk menghitung target, bukan sebagai fitur prediksi.
- Jangan mengisi jumlah antrean yang baru diketahui setelah pesanan selesai. Pesanan yang dibatalkan atau belum siap tidak dapat dipakai sebagai target tanpa penanganan khusus.
- Jika satu pesanan berisi beberapa jenis menu, **jangan pecah baris lalu menganggapnya pesanan terpisah**. Format sederhana notebook ini belum mendukung beberapa menu dalam satu pesanan; perluas representasi fitur per pesanan sebelum memakai data tersebut. Model notebook ini karena itu belum identik dengan model demo aplikasi.

In [ ]:
# 1. Pengaturan sumber data: "simulasi" atau "csv"
SUMBER_DATA = "simulasi"
PATH_CSV = "/kaggle/input/NAMA_DATASET/data_kantin.csv"  # ganti dengan lokasi file nyata


In [ ]:
# 2. Pustaka yang biasa tersedia di Kaggle
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", None)


In [ ]:
# 3. Muat data sesuai pilihan di atas
data_simulasi = SUMBER_DATA == "simulasi"
if data_simulasi:
    # Data ini dibuat untuk demonstrasi, bukan pengamatan nyata di kantin.
    rng = np.random.default_rng(2026)
    n = 360
    waktu = pd.date_range("2026-01-01 08:00:00", periods=n, freq="40min")
    menu = rng.choice(["Nasi Goreng", "Mie Ayam", "Ayam Geprek"], size=n)
    jumlah = rng.integers(1, 4, size=n)
    antrean = rng.integers(0, 12, size=n)
    durasi_menu = pd.Series(menu).map({"Nasi Goreng": 4, "Mie Ayam": 3, "Ayam Geprek": 5}).to_numpy()
    menit = np.maximum(1, 3 + durasi_menu * jumlah + 1.1 * antrean + rng.normal(0, 2, n))
    data = pd.DataFrame({
        "waktu_pesan": waktu,
        "waktu_siap": waktu + pd.to_timedelta(menit, unit="min"),
        "menu": menu,
        "jumlah": jumlah,
        "antrean_di_depan": antrean,
    })
    print("SUMBER: SIMULASI. Hasil evaluasi bukan akurasi kantin sebenarnya.")
elif SUMBER_DATA == "csv":
    data = pd.read_csv(PATH_CSV)
    if "sumber_data" in data.columns:
        sumber = data["sumber_data"].astype("string").str.strip().str.lower()
        data_simulasi = bool(sumber.eq("simulasi").any())
        if data_simulasi and not sumber.eq("simulasi").fillna(False).all():
            raise ValueError("Pisahkan data simulasi dan data nyata sebelum evaluasi.")
    print("SUMBER: CSV SIMULASI" if data_simulasi else "SUMBER: CSV; verifikasi asal data.")
else:
    raise ValueError('SUMBER_DATA harus "simulasi" atau "csv".')

print("Jumlah baris awal:", len(data))
display(data.head())


## Pemeriksaan dan penyiapan data

Data kosong, waktu tidak valid, jumlah tidak valid, antrean negatif, dan waktu siap sebelum waktu pesan dikeluarkan. Jumlah baris yang dikeluarkan ditampilkan agar bisa diperiksa kembali, bukan diam-diam dianggap benar. Untuk data nyata, selidiki terlebih dahulu alasan baris dibuang.

In [ ]:
# 4. Periksa kolom dan siapkan data
kolom_wajib = ["waktu_pesan", "waktu_siap", "menu", "jumlah", "antrean_di_depan"]
kurang = [kolom for kolom in kolom_wajib if kolom not in data.columns]
if kurang:
    raise ValueError(f"Kolom ini belum tersedia di CSV: {kurang}")

df = data[kolom_wajib].copy()
awal = len(df)
for kolom in ["waktu_pesan", "waktu_siap"]:
    df[kolom] = pd.to_datetime(df[kolom], errors="coerce")
for kolom in ["jumlah", "antrean_di_depan"]:
    df[kolom] = pd.to_numeric(df[kolom], errors="coerce")
df["menu"] = df["menu"].astype("string").str.strip()
df = df.dropna(subset=kolom_wajib)
df = df[df["menu"].ne("")]
df = df[(df["jumlah"] > 0) & (df["jumlah"] % 1 == 0)]
df = df[(df["antrean_di_depan"] >= 0) & (df["antrean_di_depan"] % 1 == 0)]
df["waktu_tunggu_menit"] = (df["waktu_siap"] - df["waktu_pesan"]).dt.total_seconds() / 60
df = df[df["waktu_tunggu_menit"] >= 0].copy()
df = df.sort_values("waktu_pesan").reset_index(drop=True)
print("Baris digunakan:", len(df))
print("Baris dikeluarkan:", awal - len(df))
if len(df) < 30:
    raise ValueError("Terlalu sedikit data valid untuk contoh pembagian. Kumpulkan setidaknya 30 baris, lalu tinjau kecukupannya.")
display(df.head())


In [ ]:
# 5. Fitur yang sudah diketahui saat pesanan diterima
df["jam_pesan"] = df["waktu_pesan"].dt.hour + df["waktu_pesan"].dt.minute / 60
df["hari_pesan"] = df["waktu_pesan"].dt.dayofweek  # Senin = 0
fitur = ["menu", "jumlah", "antrean_di_depan", "jam_pesan", "hari_pesan"]
target = "waktu_tunggu_menit"

# Bagi berdasarkan waktu UNIK, agar pesanan pada waktu yang sama tidak tersebar
# ke set yang berbeda. Data yang lebih baru dipakai untuk validasi dan uji.
# Durasi yang belum diketahui saat periode berikutnya mulai tidak ikut pelatihan.
waktu_unik = df["waktu_pesan"].drop_duplicates().sort_values().reset_index(drop=True)
if len(waktu_unik) < 10:
    raise ValueError("Perlu sedikitnya 10 waktu pemesanan berbeda untuk pembagian contoh ini.")
cut_latih = waktu_unik.iloc[int(len(waktu_unik) * 0.70)]
cut_uji = waktu_unik.iloc[int(len(waktu_unik) * 0.85)]
latih = df[(df["waktu_pesan"] < cut_latih) & (df["waktu_siap"] < cut_latih)].copy()
validasi = df[(df["waktu_pesan"] >= cut_latih) & (df["waktu_pesan"] < cut_uji)
              & (df["waktu_siap"] < cut_uji)].copy()
uji = df[df["waktu_pesan"] >= cut_uji].copy()
if min(len(latih), len(validasi), len(uji)) < 2:
    raise ValueError("Salah satu bagian terlalu kecil. Periksa distribusi waktu atau tambah data.")
for nama, bagian in [("Latih", latih), ("Validasi", validasi), ("Uji", uji)]:
    print(f"{nama}: {len(bagian)} pesanan | {bagian['waktu_pesan'].min()} s.d. {bagian['waktu_pesan'].max()}")


## Latih dan pilih model

Penyandian nama menu dipelajari **hanya dari data latih** melalui pipeline. Dua pengaturan Random Forest berikut adalah contoh kecil; pilih berdasarkan MAE validasi. Set uji belum digunakan pada tahap ini.

In [ ]:
def buat_model(regressor):
    proses = ColumnTransformer([
        ("kategori_menu", OneHotEncoder(handle_unknown="ignore"), ["menu"]),
        ("angka", "passthrough", ["jumlah", "antrean_di_depan", "jam_pesan", "hari_pesan"]),
    ])
    return Pipeline([("proses", proses), ("model", regressor)])

kandidat = [2, 5]  # min_samples_leaf
skor_validasi = []
for leaf in kandidat:
    model = buat_model(RandomForestRegressor(n_estimators=100, min_samples_leaf=leaf, random_state=2026, n_jobs=-1))
    model.fit(latih[fitur], latih[target])
    prediksi_validasi = model.predict(validasi[fitur])
    skor_validasi.append({"min_samples_leaf": leaf, "mae_validasi_menit": mean_absolute_error(validasi[target], prediksi_validasi)})

hasil_validasi = pd.DataFrame(skor_validasi).sort_values("mae_validasi_menit")
display(hasil_validasi)
leaf_terbaik = int(hasil_validasi.iloc[0]["min_samples_leaf"])
print("Pengaturan terpilih dari validasi: min_samples_leaf =", leaf_terbaik)


## Evaluasi akhir pada data uji

Setelah pengaturan dipilih, model dilatih ulang pada gabungan data latih dan validasi. Median dihitung dari gabungan data yang sama. Ketiga pendekatan dinilai pada data uji yang sama, yang lebih baru dan tidak dipakai untuk memilih pengaturan.

In [ ]:
latih_final = pd.concat([latih, validasi]).sort_values("waktu_pesan")
model_rf = buat_model(RandomForestRegressor(n_estimators=100, min_samples_leaf=leaf_terbaik, random_state=2026, n_jobs=-1))
model_linear = buat_model(LinearRegression())
baseline = DummyRegressor(strategy="median")

prediksi = {}
for nama, model in [("Median", baseline), ("Linear Regression", model_linear), ("Random Forest", model_rf)]:
    model.fit(latih_final[fitur], latih_final[target])
    prediksi[nama] = model.predict(uji[fitur])

baris_hasil = []
for nama, nilai in prediksi.items():
    baris_hasil.append({
        "metode": nama,
        "MAE (menit)": mean_absolute_error(uji[target], nilai),
        "RMSE (menit)": np.sqrt(mean_squared_error(uji[target], nilai)),
    })
hasil = pd.DataFrame(baris_hasil).sort_values("MAE (menit)")
print("Sumber data:", "SIMULASI" if data_simulasi else "CSV")
if data_simulasi:
    print("PERINGATAN: skor berikut hanya pada data simulasi, bukan akurasi kantin.")
display(hasil.round(2))

contoh = uji[["waktu_pesan", "menu", "jumlah", "antrean_di_depan", target]].copy()
contoh["prediksi_rf_menit"] = prediksi["Random Forest"]
contoh["selisih_absolut_menit"] = (contoh[target] - contoh["prediksi_rf_menit"]).abs()
for kolom in ["waktu_tunggu_menit", "prediksi_rf_menit", "selisih_absolut_menit"]:
    contoh[kolom] = contoh[kolom].round(2)
display(contoh.head(10))


## Cara membaca hasil

- **MAE** adalah rata-rata jarak absolut antara perkiraan dan waktu aktual. Misalnya MAE 3 menit pada data uji berarti selisih absolut rata-rata 3 menit, bukan jaminan setiap prediksi meleset maksimal 3 menit.
- **RMSE** juga mengukur selisih dalam menit dan memberi bobot lebih besar pada kesalahan yang besar.
- Bandingkan MAE Random Forest dengan median dan Linear Regression pada tabel yang sama. Jika Random Forest tidak lebih baik, penggunaan modelnya perlu dievaluasi; jangan menganggapnya pasti unggul.
- Hasil simulasi hanya membuktikan bahwa alur kode bisa dijalankan. Hasil data nyata bergantung pada jumlah, periode, kualitas, serta apakah data mewakili kondisi kantin. Bila data sedikit, angka tunggal belum cukup untuk menyimpulkan performa stabil.
- Jika terjadi pelayanan prioritas setelah pesanan masuk, kejadian di masa depan itu tidak dapat diketahui model saat membuat prediksi awal. Aturan pelayanan dan pencatatannya perlu dikonfirmasi sebelum ditambahkan sebagai fitur.